In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

# 1. Navegación dinámica: Buscamos el archivo pyproject.toml hacia arriba
def find_project_root(current_path, target="pyproject.toml"):
    for parent in Path(current_path).parents:
        if (parent / target).exists():
            return parent
    return None

root = find_project_root(os.getcwd())

if root:
    print(f"📂 Proyecto detectado en: {root}")
    # 2. Instalación en modo editable (-e)
    # El flag --no-deps es opcional si solo quieres registrar los cambios de archivos
    !pip install -e "{root}"
    
    print("\n✅ Instalación completada. Ya puedes importar 'legion_goes' desde cualquier celda.")
else:
    print("❌ Error: No se encontró 'pyproject.toml'. Asegúrate de estar dentro de la estructura de MAIE_tesis_github.")

📂 Proyecto detectado en: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github
Obtaining file:///home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for legion-goes (pyproject.toml) ... done
  Created wheel for legion-goes: filename=legion_goes-0.1.9-py3-none-any.whl size=2701 sha256=2cb573997f20247a21bd2324784bd71e814cbb515bee1ecb0bf2c723115accb0
  Stored in directory: /tmp/pip-ephem-wheel-cache-f3s93fw7/wheels/45/f6/84/0a48d0659fd5307d2efb8dfac8c3238f417bd9a968b770cecf
Successfully built legion-goes
  Attempting uninstall: legion-goes
    Found existing installation: legion-goes 0.1.9
    Uninstalling legion-goes-0.1.9:
      Successfully uninstalled legion-goes-0.1.9

✅ Instalación completada. Ya puedes 

In [3]:
try:
    import legion_goes
    print(f"📦 Librería 'legion_goes' lista para usar.")
except ImportError:
    print("⚠️ Instalación terminada, pero puede que necesites reiniciar el Kernel.")

📦 Librería 'legion_goes' lista para usar.


In [4]:
# Imports limpios desde la librería instalada
from legion_goes.tasks.task02_download.actions.action01_gen_plan_download import run_task02_download_action01_generate_plan
from legion_goes.tasks.task02_download.actions.action02_check_plan_download import execute_task02_download_action02_check_plan
from legion_goes.tasks.task02_download.actions.action03_run_plan_download import execute_task02_download_action03_run_download
from legion_goes.tasks.task02_download.actions.fn01_file_name_plan_download import generate_plan_download_file_path
from pathlib import Path


In [ ]:
# 1. Configuración
cfg = {
    "sat_id": "19",
    "year": "2026",
    "day": "065",#"003",
    "product_id": "ABI-L2-MCMIPF",
    "output_folder_base": Path("./data/goes_test").resolve()
}

# 2. Generar el Plan (Action 01)
run_task02_download_action01_generate_plan(**cfg)

# 3. Sincronizar Disco (Action 02)
execute_task02_download_action02_check_plan(**cfg)

# 4. Descarga Masiva (Action 03)
# Recuperamos la ruta del JSON para pasársela al Downloader
path_plan = generate_plan_download_file_path(**cfg)

execute_task02_download_action03_run_download(
    path_plan=path_plan, 
    threads=4, 
    checkpoint_n=10
)


🚀 [Action01 - Generator Plan Download]
⚠️  Status: Plan exists. Skipping.
📂 Path: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/tests/test_tasks/test_task02_download/test_actions/data/goes_test/2026/065/plan_01-download_2026_065_GOES19_EAST_ABI-L2-MCMIPF.json


🔍 [SCAN] Checking local integrity: ABI-L2-MCMIPF | 2026065 (GOES19 - EAST)
    ... scanned 100/144 items
    ... scanned 144/144 items

 ✅ Scan finished: 36/144 files found.

🔍 [PRE-CHECK] Synchronizing local inventory...
    ... scanned 100/144 items
    ... scanned 144/144 items
🧹 [CLEANUP] Scanning for temporary files...
🔍 [SCANNING] Fetching full day inventory from S3 bucket: noaa-goes19...

═══════════════════════════════════════════════════════════════════════════════════════════════
 🕒 SYSTEM TIME: 2026-03-06 10:22:16 | UTC: 09:22:16
 📅 DAILY DOWNLOAD MONITOR (Legion Goes v.0.0.1)
═══════════════════════════════════════════════════════════════════════════════════════════════
  HOUR  ║    EXP     │     S3   

In [ ]:
from pathlib import Path

# --- PARÁMETROS DE ENTRADA ---
SAT_ID = "19"
YEAR = 2026
DAY = 3
PRODUCT_ID = "ABI-L2-MCMIPF"
OUTPUT_FOLDER = Path("./data/goes_test").resolve() # Ruta absoluta para evitar líos
THREADS = 8           # Ajusta según la potencia de tu conexión en Legion
CHECKPOINT_EVERY = 5  # Guarda el JSON cada 5 archivos descargados

In [ ]:
from pathlib import Path

# 1. Define tu base (el punto actual o una ruta absoluta en Legion)
base_path = Path(".") 
output_folder_base = base_path / "data" / "plans_test"
# 2. Ejecución corregida
success = run_action01_generate_plan(
    sat_id="19",
    year="2026",
    day="003",
    product_id="ABI-L2-MCMIPF",
    # Corregido: Nombre del argumento y construcción de la ruta
    output_folder_base = output_folder_base, 
    overwrite=True
)

if success:
    print("🚀 PROCESO COMPLETADO: El JSON ha sido creado con la estructura v.1.1.0.")

In [ ]:
# =============================================================================
# JUPYTER TEST: Action 02 - Check Plan Integrity
# =============================================================================
from pathlib import Path
from legion_goes.tasks.task02_download.actions.action02_check_plan_download import execute_action_check_plan

# 1. Definir EXACTAMENTE la misma ruta que usaste para crear el plan
base_path = Path(".") 
output_folder_base = base_path / "data" / "plans_test"

# 2. Ejecución del Checker (Action 02)
# Nota: sat_id debe coincidir con el del Plan ("19")
success_check = execute_action_check_plan(
    sat_id="19",
    year=2026,
    day=3,
    product_id="ABI-L2-MCMIPF",
    output_folder_base=str(output_folder_base)
)

if success_check:
    print("\n✅ CHECKER COMPLETADO: El inventario local ha sido actualizado en el JSON.")
else:
    print("\n❌ ERROR: No se pudo realizar el chequeo. Verifica que el archivo .json existe.")

In [ ]:
from legion_goes.tasks.task02_download.actions.action03_run_plan_download import execute_action_run_download

# Usamos la misma ruta de tus pruebas anteriores
base_path = Path(".") 
output_folder_base = base_path / "data" / "plans_test"

# ¡A descargar!
success_dl = execute_action_run_download(
    sat_id="19",
    year=2026,
    day=3,
    product_id="ABI-L2-MCMIPF",
    output_folder_base=str(output_folder_base),
    threads=4,       # <--- 4 a 8 hilos es lo ideal para no saturar Legion
    overwrite=False  # Solo baja lo que el Checker marcó como faltante
)